In [0]:
import sys
import importlib
import pandas as pd

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import config
import evaluator

importlib.reload(config)
importlib.reload(evaluator)

from config import (
    MODEL_PREDICTIONS_TABLE_NAME,
    CONFIDENCE_THRESHOLDS,
    EVAL_BY_STOCK_TABLE_NAME,
    EVAL_BY_TIME_TABLE_NAME,
    EVAL_BY_CONFIDENCE_TABLE_NAME,
    EVAL_CONFIDENCE_BY_STOCK_TABLE_NAME,
)
from evaluator import ModelEvaluator

In [0]:
predictions_df = spark.table(MODEL_PREDICTIONS_TABLE_NAME).toPandas()

print("Prediction rows:", len(predictions_df))
print("Prediction cols:", len(predictions_df.columns))
predictions_df.head()

In [0]:
predictions_df = spark.table(MODEL_PREDICTIONS_TABLE_NAME).toPandas()

print("Prediction rows:", len(predictions_df))
print("Prediction cols:", len(predictions_df.columns))
predictions_df.head()

In [0]:
print(predictions_df.groupby(["Ticker", "dataset_split"]).size())
print(predictions_df["model_name"].drop_duplicates().tolist())

In [0]:
test_predictions_df = predictions_df[predictions_df["dataset_split"] == "test"].copy()

print("Test prediction rows:", len(test_predictions_df))
print("Tickers:", sorted(test_predictions_df["Ticker"].unique()))
test_predictions_df.head()

In [0]:
evalr = ModelEvaluator(confidence_thresholds=CONFIDENCE_THRESHOLDS)

In [0]:
eval_by_stock_df = evalr.evaluate_by_stock(test_predictions_df)
eval_by_stock_df

In [0]:
eval_by_time_df = evalr.evaluate_time_segments(test_predictions_df, n_segments=10)
eval_by_time_df

In [0]:
backtest_summary_df = evalr.backtest_summary(test_predictions_df)
backtest_by_stock_df = evalr.backtest_by_stock(test_predictions_df)

backtest_summary_df


In [0]:
backtest_by_stock_df

In [0]:
eval_by_confidence_df = evalr.evaluate_confidence_thresholds(test_predictions_df)
eval_by_confidence_df

In [0]:
eval_confidence_by_stock_df = evalr.evaluate_confidence_by_stock(test_predictions_df)
eval_confidence_by_stock_df

In [0]:
from config import BACKTEST_SUMMARY_TABLE_NAME, BACKTEST_BY_STOCK_TABLE_NAME

spark.createDataFrame(backtest_summary_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BACKTEST_SUMMARY_TABLE_NAME)

spark.createDataFrame(backtest_by_stock_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BACKTEST_BY_STOCK_TABLE_NAME)

In [0]:
spark.createDataFrame(eval_by_stock_df) \
    .write.mode("overwrite") \
    .saveAsTable("p2_eval_by_stock")
spark.createDataFrame(eval_confidence_by_stock_df) \
    .write.mode("overwrite") \
    .saveAsTable("p2_eval_confidence_by_stock")

spark.createDataFrame(backtest_summary_df) \
    .write.mode("overwrite") \
    .saveAsTable("p2_backtest_summary")

spark.createDataFrame(backtest_by_stock_df) \
    .write.mode("overwrite") \
    .saveAsTable("p2_backtest_by_stock")